# Day 10 — Virtual Memory & Page Replacement

Memory is split into fixed-size **pages**; a **page table** maps virtual pages to
physical frames. A miss is a **page fault** — load the page, maybe evict another.
Which one to evict is the policy. Let's watch FIFO and LRU, and meet Belady's anomaly.

## 1. Address translation

`vpn, offset = divmod(vaddr, page_size)`; the offset passes through, the page is remapped.

In [ ]:
def translate(vaddr, page_size, page_table):
    vpn, offset = divmod(vaddr, page_size)
    if vpn not in page_table: raise KeyError(f'page fault: vpn {vpn}')
    return page_table[vpn] * page_size + offset

# page 1 -> frame 2, page size 256; vaddr 356 = page 1, offset 100
print('vaddr 356 ->', translate(356, 256, {0: 5, 1: 2}), '(= frame 2 * 256 + 100)')

## 2. FIFO vs LRU on the classic reference string

In [ ]:
from collections import OrderedDict, deque

BELADY = [1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5]

def fifo(refs, k):
    frames, order, faults = set(), deque(), 0
    for p in refs:
        if p not in frames:
            faults += 1
            if len(frames) >= k: frames.discard(order.popleft())
            frames.add(p); order.append(p)
    return faults

def lru(refs, k):
    frames, faults = OrderedDict(), 0
    for p in refs:
        if p in frames: frames.move_to_end(p)
        else:
            faults += 1
            if len(frames) >= k: frames.popitem(last=False)
            frames[p] = None
    return faults

print('FIFO 3 frames:', fifo(BELADY, 3), 'faults')
print('FIFO 4 frames:', fifo(BELADY, 4), 'faults  <-- MORE frames, MORE faults!')
print('LRU  3 frames:', lru(BELADY, 3), 'faults')

## 3. Belady's anomaly, plotted

For a well-behaved policy, more frames ⇒ fewer (or equal) faults. FIFO breaks that.

In [ ]:
import matplotlib.pyplot as plt

frames = list(range(1, 7))
fifo_faults = [fifo(BELADY, k) for k in frames]
lru_faults = [lru(BELADY, k) for k in frames]

plt.figure(figsize=(7, 4))
plt.plot(frames, fifo_faults, marker='o', label='FIFO')
plt.plot(frames, lru_faults, marker='s', label='LRU')
plt.axvspan(2.5, 4.5, color='crimson', alpha=0.08)
plt.text(3.5, max(fifo_faults), 'FIFO goes UP here\n(Belady anomaly)',
         ha='center', va='top', color='crimson', fontsize=9)
plt.xlabel('number of frames'); plt.ylabel('page faults')
plt.title('More memory should mean fewer faults — FIFO disagrees')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Takeaways

- Translation splits an address into (page, offset); only the page is remapped.
- FIFO evicts by *load order* and can suffer **Belady's anomaly** — more frames, more
  faults. LRU evicts by *use order* and never does (it's a 'stack algorithm').

**Now implement** `translate`, `fifo`, and `lru` in `homework.py` (returning per-step
history), then run `pytest -q`.